# Lab Activity 7: Simulating Ubiquitin with GROMACS

Before you start: Runtime, Change runtime type, T4 GPU. Run the cells in order and wait for each one to finish. The first cell restarts the runtime once; that is expected. Instructions, checks and questions are on the Lab Activity 7 page.

## 2. Task 1: Install GROMACS in Colab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!sed -i '/cuda-version/d' /usr/local/conda-meta/pinned
!mamba install -y -q -c conda-forge cuda-version=12.9 gromacs=2026.3=nompi_cuda_h39c90b0_0 ncurses
!gmx --version | grep "GROMACS version"
!nvidia-smi -L
!pip -q install MDAnalysis biopython

## 3. Task 2: Prepare the protein

In [ ]:
import os
os.makedirs("/content/ubq", exist_ok=True); os.chdir("/content/ubq")
!wget -q https://files.rcsb.org/download/1UBQ.pdb
!grep -E "^ATOM" 1UBQ.pdb > 1ubq_clean.pdb
!gmx pdb2gmx -f 1ubq_clean.pdb -o processed.gro -water tip3p -ignh -ff amber99sb-ildn
!tail -12 topol.top

## 4. Task 3: Build the box, add water and ions

In [ ]:
!gmx editconf -f processed.gro -o box.gro -c -d 1.0 -bt cubic
!gmx solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/master/files/ions.mdp
!gmx grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr
!echo SOL | gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
!tail -5 topol.top

## 5. Task 4: Minimize

In [ ]:
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/master/files/em.mdp
!cat em.mdp
!gmx grompp -f em.mdp -c solv_ions.gro -p topol.top -o em.tpr
!gmx mdrun -v -deffnm em -nb gpu
!printf "Potential\n0\n" | gmx energy -f em.edr -o em_potential.xvg -xvg none
import numpy as np, matplotlib.pyplot as plt
d = np.loadtxt("em_potential.xvg"); plt.plot(d[:,0], d[:,1]); plt.xlabel("step"); plt.ylabel("potential energy (kJ/mol)"); plt.show()

## 6. Task 5: Equilibrate

In [ ]:
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/master/files/nvt.mdp
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/master/files/npt.mdp
!cat nvt.mdp
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
!gmx mdrun -deffnm nvt -nb gpu
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
!gmx mdrun -deffnm npt -nb gpu
!printf "Temperature\n0\n" | gmx energy -f nvt.edr -o nvt_temp.xvg -xvg none
!printf "Pressure\n0\n" | gmx energy -f npt.edr -o npt_press.xvg -xvg none
import numpy as np, matplotlib.pyplot as plt
t = np.loadtxt("nvt_temp.xvg"); p = np.loadtxt("npt_press.xvg")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t[:,0], t[:,1]); ax[0].set_xlabel("time (ps)"); ax[0].set_ylabel("temperature (K)")
ax[1].plot(p[:,0], p[:,1]); ax[1].set_xlabel("time (ps)"); ax[1].set_ylabel("pressure (bar)"); plt.show()

## 7. Task 6: Run 1 nanosecond

In [ ]:
!wget -q https://raw.githubusercontent.com/pb3lab/ibm3202/master/files/md.mdp
!sed -i 's/^nsteps.*/nsteps = 500000 ; 1 ns/' md.mdp
!grep nsteps md.mdp
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr
!gmx mdrun -deffnm md -nb gpu
!printf "Protein\nProtein\n" | gmx trjconv -s md.tpr -f md.xtc -o md_prot.xtc -pbc mol -center
!printf "Protein\nProtein\n" | gmx trjconv -s md.tpr -f md.gro -o md_prot.gro -pbc mol -center

## 8. Task 7: Measure RMSD and RMSF

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
u = mda.Universe("md_prot.gro", "md_prot.xtc")
R = rms.RMSD(u, u, select="name CA and resid 1-72", groupselections=["name CA"], ref_frame=0).run()
t, rmsd_core, rmsd_all = R.results.rmsd[:,1], R.results.rmsd[:,2], R.results.rmsd[:,3]
print("mean RMSD, last 500 ps: all %.2f A, residues 1-72 %.2f A" % (rmsd_all[t > 500].mean(), rmsd_core[t > 500].mean()))
plt.plot(t, rmsd_all, label="all 76 residues"); plt.plot(t, rmsd_core, label="residues 1 to 72")
plt.xlabel("time (ps)"); plt.ylabel("C-alpha RMSD (A)"); plt.legend(); plt.show()

In [ ]:
align.AlignTraj(u, u, select="name CA and resid 1-72", ref_frame=0, in_memory=True).run()
ca = u.select_atoms("name CA")
F = rms.RMSF(ca).run()
rmsf = F.results.rmsf
from Bio.PDB import PDBParser
s = PDBParser(QUIET=True).get_structure("x", "1UBQ.pdb")
bfac = [r["CA"].get_bfactor() for r in s[0]["A"] if "CA" in r][:76]
print("correlation RMSF vs B-factor: %.2f" % np.corrcoef(rmsf, bfac)[0, 1])
top = np.argsort(rmsf)[-5:][::-1] + 1
print("residues with the largest RMSF:", list(top))
fig, ax = plt.subplots(figsize=(10, 3.5)); ax.plot(ca.resids, rmsf, color="#0D776B", label="RMSF (A)")
ax2 = ax.twinx(); ax2.plot(ca.resids, bfac, color="#9EA3A8", label="B-factor (A^2)")
ax.set_xlabel("residue"); ax.set_ylabel("RMSF (A)"); ax2.set_ylabel("B-factor"); plt.show()

In [ ]:
!zip -q ubq_results.zip md_prot.gro md_prot.xtc md.tpr topol.top *.mdp 1UBQ.pdb
from google.colab import files
files.download("ubq_results.zip")

## 9. Task 8: Catch the AI mistake

Paste the script your AI assistant wrote (prompt on the lab page, Task 8) into the cell below, read it, run it, and compare the RMSD with your Task 7 value.

In [ ]:
# AI-written script goes here